In [ ]:
# ============================================================================
# TRADE DATACLASS AND COMPLETE MULTI-HEAD STRATEGY PIPELINE
# ============================================================================
from dataclasses import dataclass
from typing import List

import numpy as np
import pandas as pd
import tensorflow as tf

@dataclass
class Trade:
    """Professional trade representation"""
    entry_bar: int           # Bar where trade entered
    exit_bar: int            # Bar where trade exited
    entry_price: float       # Entry price at entry_bar
    exit_price: float        # Exit price at exit_bar
    trade_type: str          # 'LONG' or 'SHORT'
    bars_held: int           # Number of bars held (exit_bar - entry_bar)
    profit: float            # Absolute profit
    profit_pct: float        # Percentage profit
    exit_reason: str         # 'SPIKE', 'REV', 'TIME'

    # Calculated target levels (not executed, just reference)
    tp1_price: float = None
    tp2_price: float = None
    sl_price: float = None

    def __post_init__(self):
        """Calculate target levels after initialization"""
        if self.trade_type == 'LONG':
            self.tp1_price = self.entry_price + (self.entry_price * 0.005)   # +0.5%
            self.tp2_price = self.entry_price + (self.entry_price * 0.015)   # +1.5%
            self.sl_price = self.entry_price - (self.entry_price * 0.01)     # -1%
        else:  # SHORT
            self.tp1_price = self.entry_price - (self.entry_price * 0.005)   # -0.5%
            self.tp2_price = self.entry_price - (self.entry_price * 0.015)   # -1.5%
            self.sl_price = self.entry_price + (self.entry_price * 0.01)     # +1%

    def is_win(self) -> bool:
        return self.profit > 0

    def __repr__(self):
        return (f"Trade({self.trade_type} @{self.entry_bar}-{self.exit_bar}, "
                f"${self.entry_price:.2f}→${self.exit_price:.2f}, "
                f"{self.profit:+.2f}pts, {self.exit_reason})")

# ============================================================================
# MULTI-HEAD STRATEGY PIPELINE - COMPLETE EXECUTION
# ============================================================================
# Uses 9-output model to extract all 3 horizons
# ============================================================================

print("="*80)
print("MULTI-HEAD STRATEGY PIPELINE - 9-OUTPUT MODEL")
print("="*80)

# Prefer predictions produced by Cell 5 (model.py single source of truth).
# Fall back to manual extraction if needed.
if 'result' in globals() and ('predictions_dict' not in globals()):
    try:
        predictions_dict = result.predictions
    except Exception:
        pass

# Phase 1: Extract all 9 outputs from model
print("\n[PHASE 1] Data Extraction from 9-output model...")

if 'predictions_dict' in globals() and isinstance(predictions_dict, dict) and 'delta' in predictions_dict:
    # Canonical path: already inverse-scaled raw deltas + head outputs from model.py
    price_1min_delta = np.asarray(predictions_dict['delta']['h0']).reshape(-1)
    price_5min_delta = np.asarray(predictions_dict['delta']['h1']).reshape(-1)
    price_15min_delta = np.asarray(predictions_dict['delta']['h2']).reshape(-1)

    dir_1m = np.asarray(predictions_dict['direction_prob']['h0']).reshape(-1)
    dir_5m = np.asarray(predictions_dict['direction_prob']['h1']).reshape(-1)
    dir_15m = np.asarray(predictions_dict['direction_prob']['h2']).reshape(-1)

    var_1m = np.asarray(predictions_dict['variance']['h0']).reshape(-1)
    var_5m = np.asarray(predictions_dict['variance']['h1']).reshape(-1)
    var_15m = np.asarray(predictions_dict['variance']['h2']).reshape(-1)

    # Keep legacy variable names used later in this cell
    direction_probs = dir_5m  # Primary horizon (used only in this strategy cell)
    variance_raw = var_5m

    # Provide arrays used by downstream prints (shapes)
    price_h1 = price_5min_delta
    direction_h1 = dir_5m
    variance_h1 = var_5m

    # Also keep these available if other code expects them
    direction_h0 = dir_1m
    direction_h2 = dir_15m
    variance_h0 = var_1m
    variance_h2 = var_15m

    print("✓ Using predictions from Cell 5 (model.py)")
    print(f"  Deltas: 1m={price_1min_delta.shape}, 5m={price_5min_delta.shape}, 15m={price_15min_delta.shape}")
    print(f"  Heads:  dir(5m)={direction_probs.shape}, var(5m)={variance_raw.shape}")
else:
    # Legacy fallback: re-run model on X_test_seq and inverse-transform via target_scaler
    if 'target_scaler' not in globals():
        if 'result' in globals():
            target_scaler = result.target_scaler
        else:
            raise NameError("target_scaler is not defined; run Cell 5 first")

    if 'config' not in globals():
        if 'result' in globals():
            config = result.config
        else:
            raise NameError("config is not defined; run Cell 5 first")

    batch_size = config.BATCH_SIZE

    # Storage for all 9 outputs
    price_preds_h0, direction_preds_h0, variance_preds_h0 = [], [], []
    price_preds_h1, direction_preds_h1, variance_preds_h1 = [], [], []
    price_preds_h2, direction_preds_h2, variance_preds_h2 = [], [], []

    for i in range(0, len(X_test_seq), batch_size):
        batch_end = min(i + batch_size, len(X_test_seq))
        X_batch_tf = tf.convert_to_tensor(X_test_seq[i:batch_end], dtype=tf.float32)

        # Model returns 10 outputs: 9 predictive heads + vacuum_overflow (index 9)
        pred_outputs = list(model(X_batch_tf, training=False))
        (price_h0, direction_h0, variance_h0,
         price_h1, direction_h1, variance_h1,
         price_h2, direction_h2, variance_h2) = pred_outputs[:9]

        # Store all outputs
        price_preds_h0.append(price_h0.numpy())
        direction_preds_h0.append(direction_h0.numpy())
        variance_preds_h0.append(variance_h0.numpy())

        price_preds_h1.append(price_h1.numpy())
        direction_preds_h1.append(direction_h1.numpy())
        variance_preds_h1.append(variance_h1.numpy())

        price_preds_h2.append(price_h2.numpy())
        direction_preds_h2.append(direction_h2.numpy())
        variance_preds_h2.append(variance_h2.numpy())

    # Concatenate and trim to test size
    price_h0 = np.concatenate(price_preds_h0, axis=0)[:len(y_test)]
    direction_h0 = np.concatenate(direction_preds_h0, axis=0)[:len(y_test)]
    variance_h0 = np.concatenate(variance_preds_h0, axis=0)[:len(y_test)]

    price_h1 = np.concatenate(price_preds_h1, axis=0)[:len(y_test)]
    direction_h1 = np.concatenate(direction_preds_h1, axis=0)[:len(y_test)]
    variance_h1 = np.concatenate(variance_preds_h1, axis=0)[:len(y_test)]

    price_h2 = np.concatenate(price_preds_h2, axis=0)[:len(y_test)]
    direction_h2 = np.concatenate(direction_preds_h2, axis=0)[:len(y_test)]
    variance_h2 = np.concatenate(variance_preds_h2, axis=0)[:len(y_test)]

    # Convert to proper shapes: [N, 1] -> [N]
    direction_probs = direction_h1.ravel()  # Primary horizon
    variance_raw = variance_h1.ravel()

    # Inverse transform prices (from scaled to raw deltas)
    price_1min_delta = target_scaler.inverse_transform(price_h0).ravel()
    price_5min_delta = target_scaler.inverse_transform(price_h1).ravel()
    price_15min_delta = target_scaler.inverse_transform(price_h2).ravel()

    print(f"✓ Extracted 9 outputs:")
    print(f"  Shapes: price={price_h1.shape}, direction={direction_h1.shape}, variance={variance_h1.shape}")
    print(f"  After processing: direction_probs={direction_probs.shape}, variance_raw={variance_raw.shape}")

# Phase 2: Helper functions
print("\n[PHASE 2] Helper Functions...")

def calculate_confidence(var, eps=1e-7):
    return 1.0 / (1.0 + np.asarray(var) + eps)

def calculate_signal_strength(d, c):
    return np.asarray(d) * np.asarray(c)

def normalize_variance(v, m, s, eps=1e-7):
    return np.where(s < eps, 0.0, (v - m) / (s + eps))

def check_multi_horizon_agreement(preds, curr, thresh=0.67):
    preds = np.asarray(preds)
    up = np.sum(preds > curr)
    return max(up, len(preds) - up) / len(preds) >= thresh, max(up, len(preds) - up) / len(preds)

def detect_variance_spike(v, m, thresh=2.0, eps=1e-7):
    return v > thresh * (m + eps)

print("✓ 5 helper functions defined")

# Phase 3: Calculate metrics
print("\n[PHASE 3] Calculate Metrics...")
window = 20
var_mean = np.convolve(variance_raw, np.ones(window)/window, mode='same')
var_std = pd.Series(variance_raw).rolling(window, center=True).std().fillna(0).values
confidence = calculate_confidence(variance_raw)
signal_str = calculate_signal_strength(direction_probs, confidence)

# Multi-horizon direction signals
dir_1m = np.asarray(direction_h0).ravel()
dir_5m = np.asarray(direction_h1).ravel()
dir_15m = np.asarray(direction_h2).ravel()

# Weighted multi-horizon signal (used for plotting/entry logic)
w1 = float(getattr(config, 'LAMBDA_SHORT', 1.0))
w2 = float(getattr(config, 'LAMBDA_POINT', 1.0))
w3 = float(getattr(config, 'LAMBDA_LONG', 1.0))
denom = (w1 + w2 + w3) if (w1 + w2 + w3) != 0 else 1.0
weighted_sig = ((w1 * dir_1m) + (w2 * dir_5m) + (w3 * dir_15m)) / denom
weighted_sig = weighted_sig * confidence

print(f"✓ Metrics calculated: confidence={confidence.shape}, signal_str={signal_str.shape}")
print(f"✓ Multi-horizon directions: 1m={dir_1m.shape}, 5m={dir_5m.shape}, 15m={dir_15m.shape}")

# Phase 4: Build backtest dataframe
print("\n[PHASE 4] Building backtest dataframe...")

# Need to reconstruct close prices from deltas and last_close_test
# y_test contains deltas, last_close_test is the reference point
# actual_close[i] = last_close_test[i] + y_test[i, horizon]

# For simplicity in backtesting, use last_close_test as current price
# and predicted deltas to get future prices
close_prices = last_close_test.ravel()

backtest_data = pd.DataFrame({
    'bar': np.arange(len(close_prices)),
    'close': close_prices,
    'direction_prob': direction_probs,
    'confidence': confidence,
    'signal_strength': signal_str,
    'variance': variance_raw,
    'var_mean': var_mean,
    'var_std': var_std,
    'price_pred_1m': price_1min_delta,
    'price_pred_5m': price_5min_delta,
    'price_pred_15m': price_15min_delta,
    'dir_1m': dir_1m,
    'dir_5m': dir_5m,
    'dir_15m': dir_15m,
})

print(f"✓ Backtest dataframe created: {backtest_data.shape}")
print(f"  Columns: {list(backtest_data.columns)}")

# Phase 5: Trading strategy execution
print("\n[PHASE 5] Executing Trading Strategy...")

trades = []
position = None  # (type, entry_bar, entry_price)
max_hold = 30    # Maximum bars to hold

for bar in range(len(backtest_data) - max_hold):
    row = backtest_data.iloc[bar]
    curr_price = row['close']
    dir_prob = row['direction_prob']
    conf = row['confidence']
    sig_str = row['signal_strength']
    var_val = row['variance']
    var_m = row['var_mean']

    # Check multi-horizon agreement
    future_prices = [row['price_pred_1m'], row['price_pred_5m'], row['price_pred_15m']]
    agreement, agreement_pct = check_multi_horizon_agreement(future_prices, 0, thresh=0.67)

    # Detect variance spike
    is_spike = detect_variance_spike(var_val, var_m, thresh=2.0)

    # Entry logic
    if position is None:
        # LONG entry: high confidence UP prediction
        if dir_prob > 0.65 and conf > 0.5 and agreement and not is_spike:
            position = ('LONG', bar, curr_price)

        # SHORT entry: high confidence DOWN prediction
        elif dir_prob < 0.35 and conf > 0.5 and agreement and not is_spike:
            position = ('SHORT', bar, curr_price)

    # Exit logic
    else:
        pos_type, entry_bar, entry_price = position
        bars_held = bar - entry_bar
        exit_price = curr_price

        should_exit = False
        exit_reason = None

        # Exit on variance spike
        if is_spike:
            should_exit = True
            exit_reason = 'SPIKE'

        # Exit on reversal
        elif pos_type == 'LONG' and dir_prob < 0.45:
            should_exit = True
            exit_reason = 'REV'
        elif pos_type == 'SHORT' and dir_prob > 0.55:
            should_exit = True
            exit_reason = 'REV'

        # Exit on max hold time
        elif bars_held >= max_hold:
            should_exit = True
            exit_reason = 'TIME'

        if should_exit:
            # Calculate profit
            if pos_type == 'LONG':
                profit = exit_price - entry_price
            else:  # SHORT
                profit = entry_price - exit_price

            profit_pct = (profit / entry_price) * 100

            trade = Trade(
                entry_bar=entry_bar,
                exit_bar=bar,
                entry_price=entry_price,
                exit_price=exit_price,
                trade_type=pos_type,
                bars_held=bars_held,
                profit=profit,
                profit_pct=profit_pct,
                exit_reason=exit_reason
            )
            trades.append(trade)
            position = None

print(f"✓ Strategy executed: {len(trades)} trades generated")

# Phase 6: Performance metrics
print("\n[PHASE 6] Performance Metrics...")

if len(trades) > 0:
    wins = [t for t in trades if t.is_win()]
    losses = [t for t in trades if not t.is_win()]

    total_profit = sum(t.profit for t in trades)
    win_rate = len(wins) / len(trades) * 100
    avg_profit = total_profit / len(trades)
    avg_win = sum(t.profit for t in wins) / len(wins) if wins else 0
    avg_loss = sum(t.profit for t in losses) / len(losses) if losses else 0
    profit_factor = abs(sum(t.profit for t in wins) / sum(t.profit for t in losses)) if losses and sum(t.profit for t in losses) != 0 else float('inf')

    print("\n" + "="*80)
    print("BACKTEST RESULTS")
    print("="*80)
    print(f"Total Trades:      {len(trades)}")
    print(f"Winning Trades:    {len(wins)} ({len(wins)/len(trades)*100:.1f}%)")
    print(f"Losing Trades:     {len(losses)} ({len(losses)/len(trades)*100:.1f}%)")
    print(f"Win Rate:          {win_rate:.2f}%")
    print(f"Total Profit:      ${total_profit:,.2f}")
    print(f"Average Profit:    ${avg_profit:,.2f}")
    print(f"Average Win:       ${avg_win:,.2f}")
    print(f"Average Loss:      ${avg_loss:,.2f}")
    print(f"Profit Factor:     {profit_factor:.2f}")

    # Trade type breakdown
    long_trades = [t for t in trades if t.trade_type == 'LONG']
    short_trades = [t for t in trades if t.trade_type == 'SHORT']

    print(f"\nTrade Type Breakdown:")
    print(f"  LONG trades:     {len(long_trades)} (profit: ${sum(t.profit for t in long_trades):,.2f})")
    print(f"  SHORT trades:    {len(short_trades)} (profit: ${sum(t.profit for t in short_trades):,.2f})")

    print("\n✅ Backtesting complete!")
else:
    print("⚠️  No trades generated")

print("\n" + "="*80)

# --- CalibrationPipeline: load saved pipeline for online use ---
import os as _os
try:
    from calibration import CalibrationPipeline as _CP
    _pipeline_dir = 'calibration'
    _meta = _os.path.join(_pipeline_dir, 'pipeline_meta.json')
    if _os.path.exists(_meta):
        _cal_pipeline = _CP.load(_pipeline_dir)
    else:
        _cal_pipeline = None
        print('[calibration] No saved pipeline found - run inference.ipynb first.')
except Exception as _e:
    _cal_pipeline = None
    print(f'[calibration] Could not load pipeline: {_e}')
# Usage in trading loop:
#   p_cal = _cal_pipeline.calibrate_online(raw_prob, horizon='h1')
#   _cal_pipeline.update_online(prob_at_entry, realized_label, 'h1')
#   _cal_pipeline.save('calibration/')
# -------------------------------------------
